# Two Traders with Private Signals

In this notebook, we compute the open-loop Nash equilibrium of a game between two traders with private signals. We
compare the learned controls with the exact equilibrium.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import TwoTraders, LinearStrategies

## 1. The game

Two traders trade one asset. We denote by $\alpha^i$ the signal of trader $i$, by $S$ the price, and by $q^i$ the
inventory of trader $i$, who trades at rate $\nu^i$. These processes follow

$$
\begin{aligned}
d\alpha^i_t &= -\kappa\,\alpha^i_t\,dt + \sigma_\alpha\,dW^i_t, \\[4pt]
dS_t &= \mu_t\,dt + \sigma\,dW^S_t, \qquad \mu_t = \alpha^1_t + \alpha^2_t + \lambda(\nu^1_t + \nu^2_t), \\[4pt]
dq^i_t &= \nu^i_t\,dt ,
\end{aligned}
$$

where $W^1$, $W^2$ and $W^S$ are independent Brownian motions. Trader $i$ observes $O^i = (\alpha^i, q^i, S)$, so the
signal $\alpha^i$ is private to trader $i$. The control $\nu^i$ is adapted to the filtration generated by $O^i$, and
trader $i$ maximizes

$$
J^i = \mathbb{E}\Big[ \int_0^T \big( q^i_t\, \mu_t - a\,(\nu^i_t)^2 - \phi\,(q^i_t)^2 \big)\,dt - \psi\,(q^i_T)^2 \Big].
$$

Each trader estimates the other signal from the price. The price also moves with the trades of the other trader,
and these trades depend on the other trader's estimate of $\alpha^i$. Thus each trader estimates an estimate, and no
finite-dimensional filter closes this hierarchy. We solve the game on the Euler--Maruyama grid with $N = 50$ steps
on $[0, 1]$.

## 2. Exact equilibrium

The game is linear-quadratic and Gaussian. Thus a best response to a control linear in the observed path is itself
linear, and the equilibrium lies in the class

$$
\nu^i_{t_j} = \sum_{k \le j} \big(\beta^i_{jk,0}\,\alpha^i_{t_k} + \beta^i_{jk,1}\,S_{t_k}\big).
$$

In this class, every process is linear in the Gaussian increments of the discretised game. The expected rewards are
then computed exactly, without simulation. `LinearStrategies` computes the coefficients $\beta$ by damped
best-response iteration. Each best response holds the control of the other trader fixed as a process, so the result
is the open-loop equilibrium.

In [ ]:
game = TwoTraders(lam=1.0)
exact_solver = LinearStrategies(game)
beta = exact_solver.solve()
validation = validation_paths(game)                                # the validation paths of every Solver
exact = to_players(exact_solver.controls(beta, validation))
print(f'mean |nu| of the exact equilibrium: {np.mean([float(e.abs().mean()) for e in exact]):.4f}')

## 3. Learned and exact equilibria

The learned method computes the open-loop equilibrium by damped Picard iteration. We denote by $\mathcal{N}^i$ the
control network of trader $i$. A GRU, a recurrent network, takes the increments of $O^i$ rescaled by
$\sqrt{\Delta t}$ as input. It returns a summary $h_j \in \mathbb{R}^{32}$ of the observed path up to $t_j$. A
feedforward network of width 64 maps $(t_j, O^i_{t_j}, h_j)$ to $\nu^i_{t_j}$.

At each iteration, we simulate the game with the controls of both traders fixed. The target of trader $i$ is

$$
\frac{\Lambda^i_{t_j}}{2a} = \nu^i_{t_j} + \frac{G^i_j}{2a\,\Delta t},
$$

where $G^i_j$ is the derivative of the realised reward of trader $i$ with respect to $\nu^i_{t_j}$, computed by
automatic differentiation. The next iterate of $\mathcal{N}^i$ minimizes a mean squared error against
$(1 - \rho)\,\nu^i + \rho\,\Lambda^i / (2a)$, with damping $\rho = 0.6$. A martingale control variate is subtracted
from the target to reduce its variance.

We use 5,000 paths, 300 Adam steps per iteration and 10 iterations, and we average the controls from iteration 4.
This budget is small, so that the notebook runs in about 10 minutes on a CPU. The study in `experiments/` uses
25,000 to 100,000 paths. The figure compares the learned and exact controls on three validation paths.

In [ ]:
solver = Solver(game, damping=0.6, steps=300, batch=1000, n_paths=5000)
learned, history = solver.run(10, average_from=4, reference=exact)
print(f'relative L2 error: trader 1 {relative_l2(learned[0], exact[0]):.3f}, trader 2 {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['trader 1', 'trader 2'], ylabel=r'$\nu_t$')